## Project objectives

Build an SMS spam detection system using NLP preprocessing, TF-IDF feature extraction, and a Multinomial Naive Bayes classifier to automatically classify messages as spam or ham based on their textual content.

### Import libraries

In [77]:
import numpy as np
import pandas as pd
import re

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk import pos_tag
from nltk import RegexpParser

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

### Load the dataset

In [78]:
df = pd.read_csv("Spam_SMS.csv")

### Dataset overview

In [79]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5574 entries, 0 to 5573
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Class    5574 non-null   object
 1   Message  5574 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


In [80]:
df.head()

,Class,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


### Implement NLP

#### Download NLTK models

In [81]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_pe

True

#### Text cleaning

In [134]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    
    # lowercase
    text = text.lower()
    
    # remove punctuation
    text = re.sub(r'[^\w\s]','',text)
    
    # tokenization
    words = text.split()
    
    # stopwords removal and lemmatization
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    
    return " ".join(words)

In [135]:
df['clean_messages'] = df['Message'].apply(clean_text)

In [136]:
df['clean_messages'] = df['clean_messages'].replace('', np.nan)
df = df.dropna(subset=['clean_messages'])

In [137]:
# check

df['clean_messages'].head()

0    go jurong point crazy available bugis n great ...
1                              ok lar joking wif u oni
2    free entry 2 wkly comp win fa cup final tkts 2...
3                  u dun say early hor u c already say
4             nah dont think go usf life around though
Name: clean_messages, dtype: object

#### POS tagging

In [138]:
def extract_sentiment_words(text):
    
    if not text or text.strip() == "":
        return ""
    
    tokens = word_tokenize(text)
    tagged = pos_tag(tokens)
    
    sentiment_words = []
    
    for word, tag in tagged:
        if tag.startswith('JJ') or tag.startswith('RB'):
            sentiment_words.append(word)
            
    return " ".join(sentiment_words)

In [139]:
df['POS_messages'] = df['clean_messages'].apply(extract_sentiment_words)

In [140]:
# check

df['POS_messages'].head()

0    jurong available n great buffet amore wat
1                                     ok lar u
2                   free wkly fa final receive
3                            u early u already
4                                      nah usf
Name: POS_messages, dtype: object

#### Chunking (Phrase detection)

In [141]:
def extract_phrases(text):
    
    if not text or text.strip() == "":
        return ""
    
    tokens = word_tokenize(text)
    tagged = pos_tag(tokens)
    
    grammar = "CHUNK: {<RB>?<JJ>+}"
    parser = nltk.RegexpParser(grammar)
    
    tree = parser.parse(tagged)
    
    phrases = []
    
    for subtree in tree.subtrees():
        if subtree.label() == "CHUNK":
            phrase = " ".join(word for word, tag in subtree.leaves())
            phrases.append(phrase)
            
    return " ".join(phrases)

In [142]:
df['phrase_messages'] = df['POS_messages'].apply(extract_phrases)

In [143]:
# check

df['phrase_messages'].head()

0    available n great
1               ok lar
2           free final
3              u early
4                     
Name: phrase_messages, dtype: object

#### Apply TF-IDF

In [189]:
tfidf = TfidfVectorizer(
        max_features = 7000,
        ngram_range = (1,2),
        min_df = 1,
        max_df = 0.95,
        sublinear_tf = True
        )

#### label encoding

In [190]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['label'] = le.fit_transform(df['Class'])   # ham→0 , spam→1

In [191]:
# check

df['label'].head(3)

0    0
1    0
2    1
Name: label, dtype: int32

#### Separate target and features

In [192]:
X = df['clean_messages']
y = df['label']

#### Train test split

In [193]:
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size =0.2, random_state = 42
)

#### Transform

In [194]:
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

### Multinomial Naive Bayes model

In [195]:
model = MultinomialNB(alpha = 0.3, class_prior = [0.5,0.5])

model.fit(X_train_tfidf, y_train)

MultinomialNB(alpha=0.3, class_prior=[0.5, 0.5])

In [198]:
y_prob = model.predict_proba(X_test_tfidf)[:,1]
y_pred = (y_prob > 0.45).astype(int)

### Model evaluation

In [199]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("Accuracy: ", accuracy_score(y_test, y_pred))
print("Confusion matrix: \n", confusion_matrix(y_test, y_pred))
print("Classification report: \n", classification_report(y_test, y_pred))

Accuracy:  0.9533213644524237
Confusion matrix: 
 [[911  41]
 [ 11 151]]
Classification report: 
               precision    recall  f1-score   support

           0       0.99      0.96      0.97       952
           1       0.79      0.93      0.85       162

    accuracy                           0.95      1114
   macro avg       0.89      0.94      0.91      1114
weighted avg       0.96      0.95      0.95      1114



### Key insights

- **Overall performance:** Model achieved 95.3% accuracy, indicating strong overall classification.
- **Spam detection (main goal):** High **recall = 93%** → most spam messages are successfully detected (only 11 missed).
- **Precision trade-off: 79% spam precision** → some genuine (ham) messages are incorrectly flagged as spam (41 false alarms).
- **Ham classification:** Very reliable with 96% recall and 99% precision.
- **Balanced model:** Good **F1-score (0.85 for spam)** shows effective balance between detecting spam and avoiding false positives.
- **Conclusion:** The model prioritizes catching spam while maintaining acceptable false alerts — suitable for a practical spam filtering system.